In [1]:
from pecos.qeclib.steane.steane_simple_class import Steane
from pecos.slr import CReg, Main, SlrConverter

In [2]:
def steane_flagged_prep(basis: str = "Z", syn_rounds: int = 0, num_lqs: int = 1) -> Main:
    """Create a Steane code preparation circuit with flagged syndrome extraction.

    Args:
        basis: Preparation basis, either "Z" or "X"
        syn_rounds: Number of syndrome extraction rounds
        num_lqs: Number of logical qubits

    Returns:
        Main program object containing the circuit
    """
    s = [
        Steane(f"s_{i}", num_ancilla_qubits=4)
               for i in range(num_lqs)
    ]

    init_reject = [CReg(f"init_reject_{i}", 1) for i in range(num_lqs)]

    log_raw_out = [CReg(f"log_raw_{i}", 1) for i in range(num_lqs)]

    syn_x = [[CReg(f"syn_{i}_{j}", 3) for j in range(syn_rounds)] for i in range(num_lqs)]
    syn_z = [[CReg(f"syn_{i}_{j}", 3) for j in range(syn_rounds)] for i in range(num_lqs)]
    flags_x = [[CReg(f"flags_{i}_{j}", 3) for j in range(syn_rounds)] for i in range(num_lqs)]
    flags_z = [[CReg(f"flags_{i}_{j}", 3) for j in range(syn_rounds)] for i in range(num_lqs)]

    meas_raw_out = [CReg(f"meas_raw_{i}", 7) for i in range(num_lqs)]

    syn_raw_out = [CReg(f"syn_raw_{i}", 3) for i in range(num_lqs)]


    prog = Main(
        *init_reject,
        *log_raw_out,
        *meas_raw_out,
        *syn_raw_out,
    )

    for i in range(num_lqs):
        prog += [
            *syn_x[i],
            *syn_z[i],
            *flags_x[i],
            *flags_z[i],
        ]

    for q in range(num_lqs):
        prog += s[q]

    for q in range(num_lqs):
        prog += s[q].p(state=basis, reject=init_reject[q][0], rus_limit=1)

    for r in range(syn_rounds):
        for q in range(num_lqs):
            prog += s[q].syn_flagged(
                syn_x=syn_x[q][r],
                syn_z=syn_z[q][r],
                flags_x=flags_x[q][r],
                flags_z=flags_z[q][r],
            )

    for q in range(num_lqs):
        prog += s[q].m(meas_basis=basis, meas=meas_raw_out[q], log=log_raw_out[q][0], syn=syn_raw_out[q])

    return prog

In [3]:
prog = steane_flagged_prep(syn_rounds=4, num_lqs=3)
qasm = SlrConverter(prog).qasm()

In [4]:
print(qasm)

OPENQASM 2.0;
include "hqslib1.inc";
creg init_reject_0[1];
creg init_reject_1[1];
creg init_reject_2[1];
creg log_raw_0[1];
creg log_raw_1[1];
creg log_raw_2[1];
creg meas_raw_0[7];
creg meas_raw_1[7];
creg meas_raw_2[7];
creg syn_raw_0[3];
creg syn_raw_1[3];
creg syn_raw_2[3];
creg syn_0_0[3];
creg syn_0_1[3];
creg syn_0_2[3];
creg syn_0_3[3];
creg syn_0_0[3];
creg syn_0_1[3];
creg syn_0_2[3];
creg syn_0_3[3];
creg flags_0_0[3];
creg flags_0_1[3];
creg flags_0_2[3];
creg flags_0_3[3];
creg flags_0_0[3];
creg flags_0_1[3];
creg flags_0_2[3];
creg flags_0_3[3];
creg syn_1_0[3];
creg syn_1_1[3];
creg syn_1_2[3];
creg syn_1_3[3];
creg syn_1_0[3];
creg syn_1_1[3];
creg syn_1_2[3];
creg syn_1_3[3];
creg flags_1_0[3];
creg flags_1_1[3];
creg flags_1_2[3];
creg flags_1_3[3];
creg flags_1_0[3];
creg flags_1_1[3];
creg flags_1_2[3];
creg flags_1_3[3];
creg syn_2_0[3];
creg syn_2_1[3];
creg syn_2_2[3];
creg syn_2_3[3];
creg syn_2_0[3];
creg syn_2_1[3];
creg syn_2_2[3];
creg syn_2_3[3];
creg fla